# 04f — Features de monetización (5 features)

Nota: no hay `price_usd` en IAPs. M03 se reformula como `iap_n_total_purchases` (conteo).


In [1]:
import pandas as pd
import numpy as np
import re, time
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'data_raw'
DATA_QC_GUSTOS = ROOT / 'data' / 'data_qc_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '02_features'
DATA_QC_GUSTOS.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')

OID_RE = re.compile(r'[0-9a-f]{24}')
def clean_id(s):
    if pd.isna(s): return None
    m = OID_RE.search(str(s))
    return m.group(0) if m else None

sample = pd.read_parquet(DATA_QC_GUSTOS / 'sample_user_ids_gustos.parquet')
sample_ids = set(sample['user_id'])
N = len(sample)
features = pd.DataFrame({'user_id': sample['user_id'].values})
print(f'Sample N: {N:,}')

t0 = time.time()


Sample N: 114,412


In [2]:
# M01-M02-M03-M04: IAP flags + counts
print('[iaps]...')
t = time.time()
cons = pd.read_csv(DATA_RAW/'processed_consumables_iaps.csv', usecols=['user_id','product_id'], low_memory=False)
cons['user_id'] = cons['user_id'].apply(clean_id)
cons = cons[cons['user_id'].isin(sample_ids)]
subs = pd.read_csv(DATA_RAW/'processed_subscriptions_iaps.csv', usecols=['user_id','product_id'], low_memory=False)
subs['user_id'] = subs['user_id'].apply(clean_id)
subs = subs[subs['user_id'].isin(sample_ids)]

iap_users = set(cons['user_id']) | set(subs['user_id'])
features['is_iap_payer'] = features['user_id'].isin(iap_users).astype(int)

# M03 reformulado: count purchases (no USD disponible)
n_cons = cons.groupby('user_id').size()
n_subs = subs.groupby('user_id').size()
features['iap_n_total_purchases'] = features['user_id'].map(n_cons.to_dict()).fillna(0).astype(int) + features['user_id'].map(n_subs.to_dict()).fillna(0).astype(int)

# M04: n_distinct_products
prod = pd.concat([cons[['user_id','product_id']], subs[['user_id','product_id']]])
n_prod = prod.groupby('user_id')['product_id'].nunique()
features['iap_n_distinct_products'] = features['user_id'].map(n_prod.to_dict()).fillna(0).astype(int)
del cons, subs, prod
print(f'OK {time.time()-t:.1f}s')


[iaps]...


OK 1.0s


In [3]:
# M01 is_arena_player, M05 arena_win_rate
print('[arena]')
t = time.time()
ar = pd.read_parquet(DATA_QC_GUSTOS / '_arena_sample.parquet')
arena_total = ar.groupby('user_id').size()
features['is_arena_player'] = features['user_id'].isin(set(arena_total.index)).astype(int)

# arena_win_rate: winner_id == attacker_id?
ar['is_win'] = (ar['winner_id'] == ar['attacker_id']).astype(int)
win_rate = ar.groupby('user_id')['is_win'].mean()
features['arena_win_rate'] = features['user_id'].map(win_rate.to_dict())
del ar
print(f'OK {time.time()-t:.1f}s, arena_players: {features["is_arena_player"].sum():,}')


[arena]
OK 0.1s, arena_players: 7,172


In [4]:
assert len(features)==N and features['user_id'].is_unique
features.to_parquet(DATA_QC_GUSTOS / 'features_monetizacion.parquet', index=False)
print(f'Guardado: {features.shape}')
nan_rates = (features.isna().mean()*100).round(2)
print(nan_rates.to_string())
rep_dir = INFORMES / '04f_monetizacion'
rep_dir.mkdir(parents=True, exist_ok=True)
report = [f'# 04f - Monetización\n', f'**Shape**: {features.shape}\n', f'**Tiempo**: {time.time()-t0:.1f}s\n', '\n| Feature | %NaN |', '|---|---:|']
for f, v in nan_rates.sort_values(ascending=False).items():
    report.append(f'| {f} | {v} |')
(rep_dir / 'execution_report.md').write_text('\n'.join(report))


Guardado: (114412, 6)
user_id                     0.00
is_iap_payer                0.00
iap_n_total_purchases       0.00
iap_n_distinct_products     0.00
is_arena_player             0.00
arena_win_rate             93.73


254